In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "from pathlib import Path\n",
    "from src.data.loader import load_raw_data\n",
    "from src.data.preprocessor import encode_features, split_data\n",
    "from src.models.classifier import train_random_forest, train_xgboost, save_model\n",
    "from src.models.evaluator import evaluate_classifier, plot_confusion_matrix, plot_feature_importance\n",
    "from src.tracking.mlflow_logger import start_run, log_params, log_metrics, log_artifact, end_run\n",
    "\n",
    "# Load and preprocess\n",
    "df = load_raw_data(Path('data/raw/leads.csv'))\n",
    "df_processed, _ = encode_features(df)\n",
    "X_train, X_test, y_train, y_test, le = split_data(df_processed)\n",
    "feature_names = X_train.columns.tolist()\n",
    "\n",
    "# Train Random Forest\n",
    "rf_params = {'n_estimators': 100, 'max_depth': 5}\n",
    "start_run('RandomForest_100', 'SalesIQ')\n",
    "log_params(rf_params)\n",
    "rf_model = train_random_forest(X_train, y_train, rf_params)\n",
    "rf_metrics = evaluate_classifier(rf_model, X_test, y_test)\n",
    "log_metrics(rf_metrics)\n",
    "plot_confusion_matrix(rf_model, X_test, y_test, 'mlflow/mlartifacts/confusion_matrix_rf.png')\n",
    "log_artifact('mlflow/mlartifacts/confusion_matrix_rf.png')\n",
    "end_run()\n",
    "save_model(rf_model, 'models/rf_model.pkl')\n",
    "print(\"Random Forest done.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train XGBoost\n",
    "xgb_params = {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1}\n",
    "start_run('XGBoost_100', 'SalesIQ')\n",
    "log_params(xgb_params)\n",
    "xgb_model = train_xgboost(X_train, y_train, xgb_params)\n",
    "xgb_metrics = evaluate_classifier(xgb_model, X_test, y_test)\n",
    "log_metrics(xgb_metrics)\n",
    "plot_confusion_matrix(xgb_model, X_test, y_test, 'mlflow/mlartifacts/confusion_matrix_xgb.png')\n",
    "plot_feature_importance(xgb_model, feature_names, 'mlflow/mlartifacts/feature_importance.png')\n",
    "log_artifact('mlflow/mlartifacts/confusion_matrix_xgb.png')\n",
    "log_artifact('mlflow/mlartifacts/feature_importance.png')\n",
    "end_run()\n",
    "save_model(xgb_model, 'models/xgb_model.pkl')\n",
    "print(\"XGBoost done.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Show evaluation metrics\n",
    "print(\"Random Forest Metrics:\", rf_metrics)\n",
    "print(\"XGBoost Metrics:\", xgb_metrics)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}